# Lab 10: Check the RTC mask and guidance

**Goal:** Learn how real-time chunking (RTC) keeps a new action chunk close to the actions that the robot already committed to run. Compute the RTC v2 mask from equation 5. Compare it with the pinned upstream mask. Check the guidance term on a toy model. Read the saved development report for five runtime modes.

**Inputs:** Tracked configs, the tracked development report, and the pinned LeRobot code. **Compute:** CPU. **Cloud cost:** US$0. Start Jupyter from the repository root. Read the [lab guide](README.md).

**Limits:** This notebook does not run SmolVLA, LIBERO, or a GPU. It does not open final simulator states. The saved report has 5 development states. It does not show that RTC improves task success.

<a id="exercise"></a>

## Lab 10 exercise: Predict and check the RTC mask

Write your answer before you run each step. Use the notebook functions `eq5` and `prefix_weights`.

1. **Prediction.** For `H = 50`, `s = 10`, `d = 2`, state how many weights are above 0. State whether the weight at index 2 is above or below 0.9. Check it.
2. For `H = 6`, `L = 4`: write the soft mask for `d = 0` and for `d = 4`. Which one equals the hard mask?
3. Try `d = 5` with `L = 4`. State the result and the reason. Then state why `s = 1, d = 4` is also invalid for RTC.
4. In the toy guidance check, set `t = 0.5`. State the correction. Explain why with the factor `1 - a t`.
5. A guidance weight needs clipping near `t = 1`. Give the clip value that this course uses. State what could go wrong without it.
6. The development report shows 4/5 for `synchronous` and `rtc_hard`, and 2/5 for `rtc_soft`. Write two sentences. State what this shows and what it does not show about RTC and success.
7. The virtual delay is 100 ms. The measured p50 request time is about 177–319 ms. Convert both to ticks. State which delay the report used.
8. Read the frozen final report. Compare `naive_async` with `rtc_soft` on the same ten states. Give the successes, the discordant counts, the exact McNemar value, and the mean adjacent-action L2 values. State one limit of this comparison.


## 1. Reading assignment (20 minutes)

Read [RTC v2](https://arxiv.org/html/2506.07339v2). Read only these parts:

| Minutes | Part | Question to answer |
|---|---|---|
| 4 | Equation 3 and equation 4 | What is `Â¹`, the estimate of the finished chunk? How does `r²` set the guidance size? |
| 6 | Equation 2 | Which term is the model velocity? Which term pulls the chunk toward the old actions? Where does `β` limit it? |
| 4 | Equation 5 | Which actions get weight 1? Which get a soft weight? Which get weight 0? |
| 6 | Algorithm 1, `GuidedInference` | In which order do the mask, the noise, the error, the gradient, and the update occur? |

Stop after 20 minutes. Skip the rest of the paper.

**Symbols.** `H` is the chunk length. `s` is the number of ticks between request starts. `d` is the delay in ticks. `L = H - s` is the guided overlap. `n` is the number of denoising steps. `β` clips the guidance weight. The paper uses `τ = 0` for noise and `τ = 1` for the action. LeRobot code uses a time `t` that goes the other way. The guidance weight is the same value with `t = 1 - τ`.

**Prediction to write before you run section 2.** Use `H = 50`, `s = 10`, `d = 2`. State (a) how many weights are above 0, and (b) whether the weight at index 2 is below or above 0.9. Run section 2 to check.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import torch
import yaml
from lerobot.configs import RTCAttentionSchedule
from lerobot.policies.rtc.configuration_rtc import RTCConfig
from lerobot.policies.rtc.modeling_rtc import RTCProcessor

from vla_lab.rtc.guidance import GuidedRTCProcessor, prefix_weights
from vla_lab.runtime.virtual_scheduler import CONTROL_PERIOD_MS, latency_to_ticks

def find_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


WORKSPACE = find_root()
assert (WORKSPACE / "configs/experiments/latency.yaml").is_file(), "The required course file is missing."

H, S, D, N_STEPS, BETA = 50, 10, 2, 5, 5.0
L = H - S
assert D <= S <= H - D
print(f"H={H}, s={S}, d={D}, L=H-s={L}, n={N_STEPS}, beta={BETA}")

## 2. Write equation 5 by hand

The function below follows equation 5. It uses float64 and a loop. It does not call course code. The weight is 1 for `i < d`. It is `c(e^c - 1)/(e - 1)` for `d <= i < L`. It is 0 for `i >= L`. The value `c` is `(L - i)/(L - d + 1)`.

Two edge cases matter:

- `d = 0`: no action is committed. Every guided weight is soft.
- `d = L`: all guided weights are committed. The soft region is empty, so the soft mask equals the hard mask.

The hard mask keeps weight 1 for `i < d` and sets every other weight to 0.

In [ ]:
def eq5(horizon, overlap, delay):
    weights = np.zeros(horizon, dtype=np.float64)
    for i in range(horizon):
        if i < delay:
            weights[i] = 1.0
        elif i < overlap:
            c = (overlap - i) / (overlap - delay + 1)
            weights[i] = c * np.expm1(c) / np.expm1(1.0)
    return weights


def hard_eq(horizon, delay):
    return (np.arange(horizon) < delay).astype(np.float64)


assert np.array_equal(hard_eq(6, 2), [1, 1, 0, 0, 0, 0])
soft = eq5(H, L, D)
print("first five soft weights:", np.round(soft[:5], 4))
print("nonzero weights:", int((soft > 0).sum()), "| weight at index 2:", round(float(soft[2]), 4))
assert (soft > 0).sum() == L and soft[2] < 1.0

# d = 0: all guided weights are soft. d = L: soft equals hard.
zero = eq5(6, 4, 0)
assert zero[0] < 1.0 and np.all(np.diff(zero[:4]) < 0) and np.all(zero[4:] == 0)
full = eq5(6, 4, 4)
assert np.array_equal(full, hard_eq(6, 4)) and np.array_equal(full, [1, 1, 1, 1, 0, 0])
print("d=0  :", np.round(zero, 3))
print("d=L  :", full)

## 3. Compare the course mask with equation 5 and with the pinned upstream mask

`prefix_weights` is the course mask. `RTCProcessor.get_prefix_weights` is the pinned LeRobot code with the `EXP` schedule. Both must match equation 5. The course code keeps `d <= L <= H` and raises an error otherwise.

In [ ]:
upstream = RTCProcessor(RTCConfig(prefix_attention_schedule=RTCAttentionSchedule.EXP))

for horizon, overlap, delay in [(6, 4, 0), (6, 4, 2), (6, 4, 4), (H, L, D), (H, L, 0), (H, L, L)]:
    course = prefix_weights(horizon, overlap=overlap, delay=delay, mode="soft")
    reference = upstream.get_prefix_weights(delay, overlap, horizon)
    expected = torch.as_tensor(eq5(horizon, overlap, delay), dtype=torch.float32)
    assert torch.allclose(course, expected, atol=1e-6), (horizon, overlap, delay)
    assert torch.allclose(course, reference, atol=1e-6), (horizon, overlap, delay)
    hard = prefix_weights(horizon, overlap=overlap, delay=delay, mode="hard")
    assert torch.equal(hard, torch.as_tensor(hard_eq(horizon, delay), dtype=torch.float32))
print("Course mask = equation 5 = pinned upstream mask for 6 settings, including d=0 and d=L.")

for bad in [dict(horizon=6, overlap=4, delay=5), dict(horizon=6, overlap=7, delay=2), dict(horizon=0, overlap=0, delay=0)]:
    try:
        prefix_weights(**bad, mode="soft")
    except ValueError as error:
        print("rejected", bad, "->", error)
    else:
        raise AssertionError("Invalid RTC horizons must fail")

mask = prefix_weights(H, overlap=L, delay=D, mode="soft")
print("Prediction check: nonzero =", int((mask > 0).sum()), "| w[2] =", round(float(mask[2]), 4))

## 4. Check the guidance term on a toy model

Use a toy velocity `v(x) = a x` with `a = 2`. In LeRobot time `t`, the finished-chunk estimate is `x - t v(x) = (1 - a t) x`. Its derivative is the factor `(1 - a t)`.

The guided velocity is `v - w (1 - a t) (target - estimate) * mask`. The weight `w` is `(t² + (1-t)²) / (t (1-t))`, clipped at `β = 5`. This is the equation 2 weight written in `t`.

A correct vector-Jacobian product must contain the factor `(1 - a t)`. At `t = 0.5` the factor is 0, so the correction must vanish. A version that skips the model derivative would not vanish.

In [ ]:
A = 2.0


def guided(x, previous, t, execution=2, delay=2, mode="soft"):
    processor = GuidedRTCProcessor(RTCConfig(execution_horizon=execution, max_guidance_weight=BETA), mask_mode=mode)
    return processor.denoise_step(
        x_t=x, prev_chunk_left_over=previous, inference_delay=delay, time=t,
        original_denoise_step_partial=lambda value: A * value, execution_horizon=execution,
    )


def expected_guided(x, previous, t, execution=2, delay=2, mode="soft"):
    horizon = x.shape[1]
    overlap = horizon - execution
    weights = prefix_weights(horizon, overlap=overlap, delay=delay, mode=mode)[None, :, None]
    target = torch.zeros_like(x)
    target[:, :overlap] = previous[:, :overlap]
    estimate = (1 - A * t) * x
    weight = min(BETA, (t**2 + (1 - t) ** 2) / (t * (1 - t)))
    return A * x - weight * (1 - A * t) * (target - estimate) * weights


x = torch.linspace(0.1, 0.6, 6).reshape(1, 6, 1)
previous = torch.ones(1, 4, 1)
for mode in ("soft", "hard"):
    for t in (0.25, 0.75, 0.9):
        assert torch.allclose(guided(x, previous, t, mode=mode), expected_guided(x, previous, t, mode=mode), atol=1e-6)
assert torch.allclose(guided(x, previous, 0.5), A * x)  # Jacobian factor is 0
print("Guidance equals the closed form for hard and soft masks; correction is 0 where (1 - a t) = 0.")

# Rows at or after the overlap get no correction. The first rows do.
result = guided(x, previous, 0.25)
change = (result - A * x)[0, :, 0]
assert torch.all(change[:2] != 0) and torch.all(change[4:] == 0)

# The weight is clipped at beta near t = 1 (the noise end). The result stays finite.
assert torch.isfinite(guided(x, previous, 1.0)).all()
print("correction by row:", np.round(change.numpy(), 3))

## 5. Inspect the tracked development report

The report has 5 development states and 5 modes. All modes use one SmolVLA checkpoint (`model_sha256` below). The sampler uses `n = 5` steps. The RTC modes use `β = 5`. The scheduler uses a **virtual** delay of 2 ticks (100 ms). That delay is a setting. It is not the measured request time.

The measured request time on this host was about 175–320 ms. That time is 4 to 7 ticks. The runs did not use 4 to 7 ticks. So the report shows how each mode behaves under a fixed 2-tick delay. It does not show what the robot does under the real request time.

The reported discontinuity is the mean L2 change between all adjacent seven-value, bounded controller actions. It includes changes inside chunks and at chunk boundaries. Its unit is the dimensionless controller-action scale. A lower value does not mean a better task result.

In [ ]:
CONFIG = WORKSPACE / "configs/experiments/latency.yaml"
REPORT = WORKSPACE / "artifacts/reference/runtime/development_modes.json"
config = yaml.safe_load(CONFIG.read_text())
report = json.loads(REPORT.read_text())

MODES = ["synchronous", "naive_async", "temporal_ensemble", "rtc_hard", "rtc_soft"]
assert report["partition"] == "development" and report["state_ids"] == config["state_ids"] == [0, 1, 2, 3, 4]
assert list(report["modes"]) != [] and set(report["modes"]) == set(MODES) == set(config["modes"])
assert report["config_sha256"] == hashlib.sha256(CONFIG.read_bytes()).hexdigest()
assert report["model_sha256"] == config["learner"]["model_sha256"]
assert report["sampler_steps"] == config["runtime"]["num_steps"] == N_STEPS
assert report["virtual_delay_ticks"] == config["runtime"]["delay"] == D
assert report["virtual_delay_ms"] == D * CONTROL_PERIOD_MS == 100
assert all(report["modes"][m]["state_ids"] == [0, 1, 2, 3, 4] for m in MODES)
print("One checkpoint:", report["model_sha256"][:16], "... | states 0-4 | n =", report["sampler_steps"], "| virtual delay", report["virtual_delay_ms"], "ms")

print(f"{'mode':<18}{'success':>8}{'steps':>7}{'underfl':>8}{'discont':>9}{'p50 ms':>8}{'p95 ms':>8}{'>100ms':>9}{'peak MiB':>9}")
for mode in MODES:
    row = report["modes"][mode]
    latency = row["request_latency_ms"]
    print(f"{mode:<18}{row['successes']:>5}/{row['episodes']}{row['mean_steps']:>7.1f}{row['underflow_total']:>8}"
          f"{row['discontinuity_l2_mean']:>9.3f}{latency['p50']:>8.0f}{latency['p95']:>8.0f}"
          f"{row['requests_missed_100ms']:>5}/{row['request_count']:<3}{row['cuda_peak_allocated_bytes'] / 2**20:>9.0f}")

p50 = [report["modes"][m]["request_latency_ms"]["p50"] for m in MODES]
assert all(report["modes"][m]["requests_missed_100ms"] == report["modes"][m]["request_count"] for m in MODES)
assert 175 < min(p50) and max(p50) < 320
print(f"\nMeasured p50 range: {min(p50):.0f}-{max(p50):.0f} ms = {latency_to_ticks(min(p50))}-{latency_to_ticks(max(p50))} ticks. Virtual delay: {D} ticks.")
extra = (report["modes"]["rtc_soft"]["cuda_peak_allocated_bytes"] - report["modes"]["synchronous"]["cuda_peak_allocated_bytes"]) / 2**20
print(f"RTC peak PyTorch allocation is {extra:.0f} MiB above ordinary sampling. This excludes the renderer.")
for limit in report["limits"]:
    print("-", limit)

### Read the limits

- Success is 2 to 4 of 5. One state changes the rate by 20 points. Do not rank the modes.
- `rtc_soft` has the lowest discontinuity and the lowest success (2/5). `rtc_hard` has 4/5 and a discontinuity close to the other modes. This is one run on 5 states. It does not show that RTC improves success. It does not show that RTC hurts success either.
- Only `synchronous` has 0 underflow ticks. The other four modes have 10, which is 2 per episode. The fixed 2-tick delay causes that start-up gap.
- Every request missed the 100 ms deadline. The RTC modes take about 315–320 ms per request. The other modes take about 177 ms. The wall-clock cost of guidance is not part of the virtual delay.
- The peak memory is PyTorch allocation only. It excludes the renderer.

## 6. Verify the frozen final config without opening final states

The final matrix is frozen before any final evaluation. This cell reads two YAML files and one JSON report. It never opens a simulator state. It checks that:

1. the development report hash is the one that `latency_final.yaml` pins;
2. the final config differs from the development config only in experiment name, partition, state IDs, and the pinned report hash;
3. the final config bytes have the hash that this notebook recorded.

In [ ]:
FINAL_CONFIG = WORKSPACE / "configs/experiments/latency_final.yaml"
FROZEN_FINAL_SHA256 = "31b6935ee99b50512c9cd23d39050c8964f8dee7fdb824f3feb190397e466066"
FROZEN_DEVELOPMENT_SHA256 = "87cc4e421d8ded755d6e03cd9c8186a19d502483f2d4ed99d165f7c6fd64bae5"
FROZEN_REPORT_SHA256 = "a2ade9ddd845e2b553b8095fd7492697ea03d88823c9f625ad9f919dbb0091fa"


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


final = yaml.safe_load(FINAL_CONFIG.read_text())
assert sha256(FINAL_CONFIG) == FROZEN_FINAL_SHA256
assert sha256(CONFIG) == FROZEN_DEVELOPMENT_SHA256 == report["config_sha256"]
assert sha256(REPORT) == FROZEN_REPORT_SHA256 == final["development_report_sha256"]

changed = {key for key in set(config) | set(final) if config.get(key) != final.get(key)}
assert changed == {"experiment", "partition", "state_ids", "development_report_sha256"}, changed
assert final["state_ids"] == list(range(20, 30))
assert not set(final["state_ids"]) & set(config["state_ids"])
assert final["modes"] == MODES and final["runtime"] == config["runtime"] and final["learner"] == config["learner"]
print("Final config hash:", FROZEN_FINAL_SHA256)
print("Final states", final["state_ids"][0], "to", final["state_ids"][-1], "; development states 0 to 4; no overlap.")
print("Final config uses the same checkpoint, runtime (H=50, s=10, d=2, n=5), and modes as development.")

current = {path: sha256(WORKSPACE / path) for path in report["source_file_hashes"]}
print("Development report matches current source files:", current == report["source_file_hashes"])

final_reports = sorted(p.name for p in (WORKSPACE / "artifacts/reference/runtime").glob("*final*.json"))
print("Tracked final report files:", final_reports or "none")

## 7. Read the frozen final report

This cell reads the tracked report. It does not open a simulator state. Check every state ID, the frozen recipe hash, the model hash, and all 50 attempts. The table is a small mechanism study. Do not use its outcome to change the frozen recipe.

In [ ]:
FINAL_REPORT = WORKSPACE / "artifacts/reference/runtime/final_modes.json"
final_report = json.loads(FINAL_REPORT.read_text())
assert final_report["partition"] == "final"
assert final_report["state_ids"] == final["state_ids"] == list(range(20, 30))
assert final_report["config_sha256"] == FROZEN_FINAL_SHA256
assert final_report["development_report_sha256"] == FROZEN_REPORT_SHA256
assert final_report["model_sha256"] == report["model_sha256"]
assert final_report["completed_episodes"] == 50 and final_report["runner_errors"] == 0
assert set(final_report["modes"]) == set(MODES)
print(f"{'mode':<18}{'success':>8}{'underfl':>9}{'adjacent L2':>14}{'p50 ms':>9}")
for mode in MODES:
    row = final_report["modes"][mode]
    assert row["state_ids"] == final["state_ids"] and row["episodes"] == 10
    print(f"{mode:<18}{row['successes']:>5}/10{row['underflow_total']:>9}"
          f"{row['discontinuity_l2_mean']:>14.3f}{row['request_latency_ms']['p50']:>9.0f}")
print("Paired naive_async vs rtc_soft:", final_report["paired"]["naive_async_vs_rtc_soft"])
print("Limit: ten final states and one checkpoint do not establish a stable ranking.")

## 8. Optional: run the five-mode experiment on a local GPU

You do not need this step to finish the lab. It needs a CUDA GPU, the LIBERO assets, and the pinned checkpoint and VLM under `.local-data`. It runs 25 episodes (5 modes × 5 development states) and reuses saved episodes. Run it from the repository root in a terminal:

```bash
.venv/bin/python -m vla_lab.experiments.runtime_modes \
  --config configs/experiments/latency.yaml \
  --output .local-data/runs/runtime-modes-development
```

Add `--modes rtc_soft --state-ids 0` for one short run. Use only `latency.yaml`. Do not point this command at `latency_final.yaml`; the final states are for the frozen evaluation only.

The command prints a summary with the same fields as the tracked report. Your timing values depend on your GPU and host.

## 9. What you proved and what you did not prove

You proved these points:

- The course mask equals equation 5 and the pinned upstream mask, including `d = 0` and `d = L`.
- Invalid horizons fail.
- The guidance term includes the model derivative and is finite when the weight is clipped.
- The tracked reports use one checkpoint, `n = 5`, and a virtual delay of 100 ms. The RTC modes use `β = 5`.
- The final config was frozen and pinned to the development report before the final run. The tracked final report has all 50 attempts.

You did not prove these points:

- That RTC improves task success. The final report has ten states and shows no stable ranking.
- That 100 ms virtual delay is the real delay. The measured request time was about 175–320 ms.
- That a lower discontinuity gives a better robot result.
- A stable ranking from the ten final states, or any result for other checkpoints or tasks.

Next, read the [exercise](#exercise). Read the [solution guide](../../solutions/10-rtc.md) after you save your answer.